# CLP Data Generation Pipeline

This notebook covers the complete data generation pipeline for the CLP Transformer model. The process is divided into four stages: generating synthetic problem instances, executing search rollouts to build solution trajectories, vectorizing the states/actions using adapters, and compiling the final datasets for training.

## 1. Instance Generation

Generates a set of synthetic CLP instances. Each instance is defined by a specific number of unique block types (`n_types`) and initialized with a random seed for reproducibility.

In [8]:
from instances.generation import generate_instances

filename = "example_M30.txt"
n_instances = 100
n_types = 30
seed = 42

generate_instances(filename, n_instances, n_types, seed)

## 2. Beam Search Rollouts

Executes parallel beam search rollouts on the generated instances to create solution trajectories. The search explores candidate actions using a defined beam width (`w`). Additionally, the `min_fr` parameter controls the initial block generation stage, setting the minimum volume fill rate required to create compact blocks of boxes.

In [1]:
from data.generation import run_instances_parallel

filename = "example_M30.txt"
w = 8
num_actions = 64
max_workers = None
min_fr = 0.98

run_instances_parallel(filename, w, num_actions, min_fr, double_effort=False, max_workers=max_workers)

Progreso: 100.00% (100/100)
Salida guardada en: /home/oscar/Escritorio/CLP-Framework/outputs/example_M30


## 3. Data Vectorization

Applies the `InputAdapter` and `OutputAdapter` to transform the raw heuristic trajectories into structured, vectorized features (environment states) and one-hot labels (target actions) compatible with the Transformer architecture.

In [2]:
from data.generation import generate_data
from data.adapters.input.v1 import InputAdapterV1
from data.adapters.output.action_adapter import ActionAdapter

folder = "example_M30"
input_adapter = InputAdapterV1(max_blocks=10000, max_pblocks=64, max_actions=64)
output_adapter = ActionAdapter(max_actions=64)
min_blocks = 1000
min_actions = 4
max_actions = 64

generate_data(folder, input_adapter, output_adapter, min_blocks, min_actions, max_actions)

Datos guardados en: /home/oscar/Escritorio/CLP-Framework/data/example_M30.data (Tamaño 13606)


## 4. Dataset Preprocessing

Processes the vectorized trajectory data into final dataset formats ready for model training. This step applies size constraints (`max_size`) and predefined data splits (`cuts`).

In [3]:
from data.preprocessing import generate_datasets

filenames = ["example_M30.data"]
basename = "example_M30"
cuts = [1, 1, 8, 64]
max_size = 100

generate_datasets(filenames, basename, cuts, max_size)

Procesando corte: [1, 1]
Dataset guardado en: /home/oscar/Escritorio/CLP-Framework/data/datasets/example_M30_1-1.data (Tamaño 100)
Procesando corte: [2, 8]
Dataset guardado en: /home/oscar/Escritorio/CLP-Framework/data/datasets/example_M30_2-8.data (Tamaño 100)
Procesando corte: [9, 64]
Dataset guardado en: /home/oscar/Escritorio/CLP-Framework/data/datasets/example_M30_9-64.data (Tamaño 100)
